# Phase 2a — Explore Baseline Panel

Eyeball check on the synthetic panel produced by `scripts/generate_mock_data.py`.

Four cells:
1. Load the seed-42 parquet and describe its shape.
2. Baseline price paths for the tier-median constituent of each tier — verifies drift + step events + bidirectional behaviour at the tier level.
3. Per-contributor bias deviation on `openai/gpt-5-mini` (broadest coverage, 9 contributors) — verifies that each contributor's systematic bias lands near its configured `price_bias_pct`.
4. Histogram of daily step-event counts across all 16 models — verifies the Poisson generator produced the design-specified rate (~0.175 events/day) at the aggregate level.

Not auto-executed on commit. Run all cells to see charts.

## Cell 1 — Load and describe the panel

In [ ]:
from pathlib import Path

import pandas as pd

PANEL_PATH = Path("../data/raw/mock_panel_clean_seed42.parquet")
panel = pd.read_parquet(PANEL_PATH)

print(f"Shape: {panel.shape}")
print(f"Date range: {panel['observation_date'].min().date()} to {panel['observation_date'].max().date()}")
print(f"Contributors: {panel['contributor_id'].nunique()}, Models: {panel['constituent_id'].nunique()}")
panel.head()

## Cell 2 — Tier-median baseline price paths

One representative constituent per tier (picked as the tier-median baseline price, not the cheapest or most expensive):
- **F**: `openai/gpt-5` at $40/Mtok output (tier median $57.50)
- **S**: `mistral/mistral-large-3` at $6/Mtok output (tier median $4.50)
- **E**: `deepseek/deepseek-v3-2` at $1.00/Mtok output (tier median $0.80)

Plotting the raw baseline (pre-contributor-noise) via `generate_baseline_prices` so step events are visible as clean jumps.

Expected visual: clear downward drift punctuated by discrete step events; occasional upward moves (25% of step events) prove the bidirectional design.

In [ ]:
from datetime import date

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tprr.config import load_all
from tprr.mockdata.pricing import generate_baseline_prices

cfg = load_all()
baseline = generate_baseline_prices(
    cfg.model_registry, date(2025, 1, 1), date(2026, 4, 23), seed=42
)

TIER_MEDIANS = [
    ("TPRR_F", "openai/gpt-5", 40.0),
    ("TPRR_S", "mistral/mistral-large-3", 6.0),
    ("TPRR_E", "deepseek/deepseek-v3-2", 1.0),
]

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[
        f"{tier}: {cid} — baseline ${base:.2f}/Mtok"
        for tier, cid, base in TIER_MEDIANS
    ],
    vertical_spacing=0.08,
)

for row, (_tier, cid, base) in enumerate(TIER_MEDIANS, start=1):
    series = baseline[baseline["constituent_id"] == cid].sort_values("date")
    fig.add_trace(
        go.Scatter(
            x=series["date"],
            y=series["baseline_output_price_usd_mtok"],
            mode="lines",
            line=dict(width=1, color="steelblue"),
            showlegend=False,
        ),
        row=row,
        col=1,
    )
    fig.add_hline(
        y=base,
        line_dash="dash",
        line_color="gray",
        opacity=0.4,
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text="$/Mtok output", row=row, col=1)

fig.update_layout(
    height=720,
    title="Baseline output price paths — tier-median constituents (seed 42)",
    margin=dict(t=70, b=40),
)
fig.show()

## Cell 3 — Per-contributor bias deviation on `openai/gpt-5-mini`

`gpt-5-mini` is covered by 9 of 10 contributors (the broadest coverage in the panel). Plotting each contributor's percentage deviation from the baseline over the full backtest reveals the systematic bias on each contributor's submissions.

Expected visual: 9 lines, each oscillating around its configured `price_bias_pct` — should span from sirius at -2.0% to rigel at +1.5%.

In [ ]:
TARGET = "openai/gpt-5-mini"

target_baseline = (
    baseline[baseline["constituent_id"] == TARGET]
    .sort_values("date")
    .set_index("date")["baseline_output_price_usd_mtok"]
)

panel_target = panel[panel["constituent_id"] == TARGET].copy()
panel_target["baseline"] = panel_target["observation_date"].map(target_baseline)
panel_target["deviation_pct"] = 100.0 * (
    panel_target["output_price_usd_mtok"] / panel_target["baseline"] - 1.0
)

fig = go.Figure()
for cid in sorted(panel_target["contributor_id"].unique()):
    rows = panel_target[panel_target["contributor_id"] == cid].sort_values(
        "observation_date"
    )
    fig.add_trace(
        go.Scatter(
            x=rows["observation_date"],
            y=rows["deviation_pct"],
            mode="lines",
            name=cid,
            line=dict(width=1),
            opacity=0.75,
        )
    )

fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)
fig.update_layout(
    height=520,
    title=f"Per-contributor price deviation from baseline — {TARGET}",
    xaxis_title="Date",
    yaxis_title="Deviation from baseline (%)",
    legend=dict(orientation="h", y=-0.22, x=0),
    margin=dict(t=60, b=120),
)
fig.show()

## Cell 4 — Daily step-event count histogram

For each day in the 478-day window, count the number of constituents (out of 16) that experienced a step event that day. A "step event" is detected at the baseline level as `|day-over-day return| > 4%` — this threshold reliably separates the 5%+ step events from 3σ daily noise (max ~1.2% for the E tier).

**Theoretical expectation**: Poisson with rate = sum(tier_rate × n_models_in_tier) / 365 = (6×3 + 4×4 + 6×5) / 365 ≈ **0.175 events/day** across all 16 models. Distribution should be right-skewed with most days at zero events, a long thin tail toward higher counts.

In [ ]:
baseline_sorted = baseline.sort_values(["constituent_id", "date"]).reset_index(drop=True)
baseline_sorted["return"] = baseline_sorted.groupby("constituent_id")[
    "baseline_output_price_usd_mtok"
].pct_change()
baseline_sorted["event"] = baseline_sorted["return"].abs() > 0.04
events_per_day = baseline_sorted.groupby("date")["event"].sum().astype(int)

observed_mean = float(events_per_day.mean())
theoretical_mean = (6 * 3 + 4 * 4 + 6 * 5) / 365.0

fig = go.Figure(
    go.Histogram(
        x=events_per_day.values,
        xbins=dict(start=-0.5, end=int(events_per_day.max()) + 0.5, size=1),
        marker_color="steelblue",
    )
)
fig.add_annotation(
    text=(
        f"Observed mean: {observed_mean:.3f} events/day<br>"
        f"Theoretical (Poisson): {theoretical_mean:.3f} events/day"
    ),
    xref="paper",
    yref="paper",
    x=0.98,
    y=0.95,
    xanchor="right",
    yanchor="top",
    showarrow=False,
    bgcolor="white",
    bordercolor="gray",
    borderpad=6,
)
fig.update_layout(
    height=420,
    bargap=0.1,
    title="Daily step-event count across all 16 models (478-day window)",
    xaxis_title="Step events per day",
    yaxis_title="Number of days",
    margin=dict(t=60, b=60),
)
fig.show()